In [41]:
#Daten importieren
import pandas as pd
df = pd.read_csv("../data/ab_ag.tsv", sep="\t")

#dimension
print(df.shape)

#zeigen welche spalten es gibt
print(df.columns)



(5523, 30)
Index(['pdb', 'Hchain', 'Lchain', 'model', 'antigen_chain', 'antigen_type',
       'antigen_het_name', 'antigen_name', 'short_header', 'date', 'compound',
       'organism', 'heavy_species', 'light_species', 'antigen_species',
       'authors', 'resolution', 'method', 'r_free', 'r_factor', 'scfv',
       'engineered', 'heavy_subclass', 'light_subclass', 'light_ctype',
       'affinity', 'delta_g', 'affinity_method', 'temperature', 'pmid'],
      dtype='object')


In [42]:
# Nur die gewünschten Spalten behalten
df_cleaned = df[["pdb",	"Hchain", "Lchain", "antigen_species", "heavy_subclass", "light_subclass", "antigen_type", "antigen_name", "organism", "resolution", "light_ctype"]]

#zeilen die noch übrig sind
print(df_cleaned.columns)

Index(['pdb', 'Hchain', 'Lchain', 'antigen_species', 'heavy_subclass',
       'light_subclass', 'antigen_type', 'antigen_name', 'organism',
       'resolution', 'light_ctype'],
      dtype='object')


In [43]:
#alle Zeilen in denen NaN vorkommt entfernen 
df_cleaned = df_cleaned.dropna()

#zum überprüfen
print("Vorher:", len(df))
print("Nachher:", len(df_cleaned))



Vorher: 5523
Nachher: 5257


In [44]:
#dimension anschauen
print(df_cleaned.shape)


(5257, 11)


In [45]:
# Zuerst sicherstellen, dass die Spalte 'resolution' numerisch ist
df_cleaned['resolution'] = pd.to_numeric(df_cleaned['resolution'], errors='coerce')

# Step 1: Dann nur Zeilen mit resolution <= 3.0 behalten
df_cleaned = df_cleaned[df_cleaned['resolution'] <= 3.0]

#dimension anschauen
print(df_cleaned.shape)

(2799, 11)


In [46]:
#Step 2: was gibt es alles in der Spalte Antigen typen
print(df_cleaned['antigen_type'].value_counts())
#keine weiter filterung notwendig

antigen_type
protein                        2523
protein | protein               217
protein | protein | protein      28
protein | peptide                14
peptide | protein                 7
peptide                           6
protein | protein | peptide       2
peptide | protein | protein       1
protein | peptide | protein       1
Name: count, dtype: int64


In [50]:
# Step 3: Entferne Zeilen mit fehlender Hchain, Lchain oder antigen_name
#wurde davor schon durch NaN entfernt

df_cleaned.head()

,pdb,Hchain,Lchain,antigen_species,heavy_subclass,light_subclass,antigen_type,antigen_name,organism,resolution,light_ctype
3,8uzp,H,L,mus musculus,IGHV1,IGLV1,protein,stem_mimetic_01,Homo sapiens; Mus musculus,2.711,Lambda
4,8uzp,A,B,mus musculus,IGHV1,IGLV1,protein,stem_mimetic_01,Homo sapiens; Mus musculus,2.711,Lambda
5,8veb,G,I,influenza a virus,IGHV4,IGKV1,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.970,Kappa
6,8veb,H,L,influenza a virus,IGHV4,IGKV1,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.970,Kappa
8,8ved,H,L,influenza a virus,IGHV4,IGKV2,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.980,Kappa


In [48]:
# Steop 4: remove redundant structures

# Alle eindeutigen PDB-IDs aus der Spalte 'pdb' holen (ohne nochmal klein zu machen)
pdb_ids = df_cleaned['pdb'].unique().tolist()

print(f"Es sind {len(pdb_ids)} einzigartige PDB-IDs zum Herunterladen.")


Es sind 1480 einzigartige PDB-IDs zum Herunterladen.


In [49]:
from Bio.PDB import PDBList
import os

pdbl = PDBList()

# Ordner, in dem die PDB-Dateien gespeichert werden sollen
download_folder = "../data/pdb_files"

# Ordner erstellen, falls nicht vorhanden
os.makedirs(download_folder, exist_ok=True)

# Alle PDB-Dateien herunterladen
for pdb_id in pdb_ids:
    print(f"Lade PDB {pdb_id} herunter...")
    pdbl.retrieve_pdb_file(pdb_id, pdir=download_folder, file_format='pdb')


Lade PDB 8uzp herunter...
Structure exists: '../data/pdb_files/pdb8uzp.ent' 
Lade PDB 8veb herunter...
Structure exists: '../data/pdb_files/pdb8veb.ent' 
Lade PDB 8ved herunter...
Structure exists: '../data/pdb_files/pdb8ved.ent' 
Lade PDB 9dpc herunter...
Structure exists: '../data/pdb_files/pdb9dpc.ent' 
Lade PDB 9dru herunter...
Structure exists: '../data/pdb_files/pdb9dru.ent' 
Lade PDB 9ds1 herunter...
Structure exists: '../data/pdb_files/pdb9ds1.ent' 
Lade PDB 9mer herunter...
Structure exists: '../data/pdb_files/pdb9mer.ent' 
Lade PDB 9mev herunter...
Structure exists: '../data/pdb_files/pdb9mev.ent' 
Lade PDB 8rmx herunter...
Structure exists: '../data/pdb_files/pdb8rmx.ent' 
Lade PDB 8rmy herunter...
Structure exists: '../data/pdb_files/pdb8rmy.ent' 
Lade PDB 8ykt herunter...
Structure exists: '../data/pdb_files/pdb8ykt.ent' 
Lade PDB 9azr herunter...
Structure exists: '../data/pdb_files/pdb9azr.ent' 
Lade PDB 9azt herunter...
Structure exists: '../data/pdb_files/pdb9azt.ent' 

In [51]:
#cluster VL and VH

In [52]:
df_cleaned


,pdb,Hchain,Lchain,antigen_species,heavy_subclass,light_subclass,antigen_type,antigen_name,organism,resolution,light_ctype
3,8uzp,H,L,mus musculus,IGHV1,IGLV1,protein,stem_mimetic_01,Homo sapiens; Mus musculus,2.711,Lambda
4,8uzp,A,B,mus musculus,IGHV1,IGLV1,protein,stem_mimetic_01,Homo sapiens; Mus musculus,2.711,Lambda
5,8veb,G,I,influenza a virus,IGHV4,IGKV1,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.970,Kappa
6,8veb,H,L,influenza a virus,IGHV4,IGKV1,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.970,Kappa
8,8ved,H,L,influenza a virus,IGHV4,IGKV2,protein,hemagglutinin,Homo sapiens; Influenza A virus,2.980,Kappa
...,...,...,...,...,...,...,...,...,...,...,...
5517,3q1s,H,L,homo sapiens,IGHV4,IGKV3D,protein,interleukin-22,HOMO SAPIENS,2.150,Kappa
5518,4zfg,H,L,homo sapiens,IGHV3,IGKV1,protein,angiopoietin-2,HOMO SAPIENS,2.270,Kappa
5519,5sy8,H,L,synthetic construct,IGHV3,IGLV3,protein,10e8 epitope scaffold t117v2,HOMO SAPIENS; SYNTHETIC CONSTRUCT,1.620,Lambda
5520,3idx,H,L,human immunodeficiency virus 1,IGHV3,IGKV1,protein,hiv-1 hxbc2 gp120 core,HUMAN IMMUNODEFICIENCY VIRUS 1; HOMO SAPIENS,2.500,Kappa


In [ ]:
df_clean 